# The super social network
In this notebook, you explore various techniques to gain insight into a social network. You will examine how widespread the network is, which influencers are in the network, and which communities are in the network.

Before you start, import the necessary libraries.

In [47]:
import csv
import numpy as np
import networkx as nx

## Loading the social network
In the folder *dataset* you can find a file named *hero-network.csv*. Each line of this file contains two superhero names. Such a pair was added to the file when the two heroes appeared together in a comic book.

**Assignment**:- Open the file *hero-network.csv* in the folder *dataset*;- View the file.- Do you recognize any heroes?- Are there any heroes you don't recognize?

With the following code cell, we load the contents of the CSV file into a NumPy array.

In [ ]:
# Lees het csv bestand.
data = None
with open('./dataset/hero-network-small-sample.csv', newline='') as csvfile:
    data = np.array(list(csv.reader(csvfile)))
    
# Verwijder de eerste rij, deze bevat de kolomnamen.
data = data[1:]

## Gaining insight into the dataset
Before you start analyzing a dataset, it's a good idea to get a sense of the data contained in that dataset. Below, we explore a number of simple properties of the dataset.

Run the following code cell to print 20 random hero pairs.

In [ ]:
# Print 20 radom rijen van de data.
print(data[np.random.randint(0, len(data), 20)])

By printing the shape of our numpy array, we learn how many hero pairs are in the dataset.

In [ ]:
# Druk het formaat af van de dataset.
print(f"Er zitten {data.shape[0]} duo's in de dataset.")

When you look at the csv file, you will see that many heroes appear multiple times in the dataset. With the following code cell, we count how many unique heroes appear in the dataset.

In [ ]:
# Verzamel de unieke helden in een lijst.
helden = np.unique(data)
print(f"Er zitten {len(helden)} unieke helden in de dataset.")

We can also print the heroes alphabetically.

In [ ]:
helden_alfabetisch = list(np.sort(helden))
for held in helden_alfabetisch[:20]:
    print(f"{held}")

To determine in how many comics two heroes come into contact with each other, we count the number of times each pair appears in the dataset. Note that the order in which the heroes' names appear in the dataset is not important. The pair (Captain America, The Hulk) is the same as (The Hulk, Captain America).

In [71]:
# Maak een dictionary die het aantal interacties per koppel helden bijhoudt.

# Initialiseer de dictionary.
interacties = {}

# Overloop alle heldenkoppels
for rij in data:
    # Sorteer de namen van het koppel zodat de volgorde niet uitmaakt.
    heldenpaar = tuple(sorted(rij))
    # Voeg het heldenpaar toe aan de dictionary of verhoog het aantal interacties.
    if heldenpaar in interacties:
        interacties[heldenpaar] += 1
    else:
        interacties[heldenpaar] = 1
        

Below we print how often the first 10 hero pairs appear in the dataset.

In [ ]:
for paar in list(interacties.keys())[:20]:
    print(f"{paar[0]} en {paar[1]} interageren {interacties[paar]} keer.")

## Constructing a graph
To build a graph, we use the networkx library. This library makes it easy to perform analyses on graphs.

First of all, we create a graph object. We'll add our nodes and edges to this object.

In [73]:
# Maak een graafobject aan.
graaf = nx.Graph()

We can easily add our heroes as nodes to the graph. For that, we use the function `add_nodes_from`. We pass a list to this function. The function will then add a node for each element in the list.

In [74]:
# Voeg de helden toe als knopen.
graaf.add_nodes_from(helden)

Now we only need to add the edges to the graph. For that, we can use our dictionary of interactions. Each interaction corresponds to an edge between the nodes of the two heroes in the pair. The weight of the edge is the number of interactions that pair has.

In [75]:
# Voeg de interacties toe als gewogen bogen.
for paar in interacties:
    graaf.add_edge(paar[0], paar[1], weight=interacties[paar])

## Finding influencers
As you saw earlier in the learning path, you can identify important people in a social network based on their **degree centrality**. Networkx has a simple method to determine the degree centrality of each node. The code below prints the 10 heroes with the highest degree centrality.

In [ ]:
graadcentraliteit = nx.degree_centrality(graaf)
gesorteerde_graadcentraliteit = sorted(graadcentraliteit.items(), key=lambda x: x[1], reverse=True)

for held, graad in gesorteerde_graadcentraliteit[:10]:
    print(f"{held} heeft een graadcentraliteit van {graad}.")

**Assignment**: In addition to degree centrality, you can also compute eigenvector centrality with the function `eigenvector_centrality`. Use the code cell below to print the 10 most popular heroes according to eigenvector centrality.

**Assignment**: Compare the most popular heroes according to eigenvector and degree centrality. What differences do you see?

## Distribution of the network
The spread of the network is the maximum shortest distance between any two heroes in the network. For each pair of nodes, we compute the length of the shortest path between these two nodes using Dijkstra's algorithm. Among all these distances, we take the longest distance as the network's *diameter*. This diameter tells us something about how spread out the network is. The smaller the diameter, the more strongly the nodes in the network are connected to each other.



**Note that the code in the next cell can take a lot of time. After all, the computer must calculate the distance between every pair of nodes in the graph; there are about 2220 x 2220 = 4928400 possible pairs.**

In [ ]:
# Calculate the maximal shortest path between any two nodes
diameter = nx.diameter(graaf)
print(f"The diameter van de graaf is {diameter}")

## Search communities
In addition to the influencers in the network and the spread of the network, we can also look for communities within the network. In a social network, a community often corresponds to a specific group of friends. For example, people who know each other from the sports club.

To detect communities, we use the Louvain method. This is an algorithm that will iteratively search for communities within the network. The algorithm does this by calculating the network’s *modularity* and maximizing it step by step. Here we use a special library that already contains an implementation of the algorithm.

First we install the library.

In [ ]:
!pip install python-louvain

Then we import the library.

In [85]:
import community

With this library, we can easily search for "friend groups" in our network.

In [86]:
vriendengroepen = community.best_partition(graaf)

First of all, we look at how many friend groups the algorithm has found.

In [ ]:
# Druk het aantal vriendengroepen in het netwerk af.
print(f"Er zijn {len(set(vriendengroepen.values()))} vriendengroepen in het netwerk.")

We can also see how many heroes are in each friend group.

In [ ]:
# Druk voor elke vriendengroep het aantal leden af.
for vriendengroep in set(vriendengroepen.values()):
    leden = [held for held, groep in vriendengroepen.items() if groep == vriendengroep]
    print(f"Vriendengroep {vriendengroep} heeft {len(leden)} leden.")

To find the most popular heroes in each friend group, we can rank the heroes within a friend group according to degree centrality. The cell below prints the 5 most popular heroes for each friend group.

In [ ]:
# Druk de eerste 5 leden van elke vriendengroep af gerangschikt volgens graadcentraliteit.
for vriendengroep in set(vriendengroepen.values()):
    leden = [held for held, groep in vriendengroepen.items() if groep == vriendengroep]
    graadcentraliteit_van_leden = {held: graadcentraliteit[held] for held in leden}
    gesorteerde_leden = sorted(graadcentraliteit_van_leden.items(), key=lambda x: x[1], reverse=True)
    print(f"Vriendengroep {vriendengroep} heeft de volgende top 5 leden:")
    for held, graad in gesorteerde_leden[:5]:
        print(f"    - {held} met een graadcentraliteit van {graad}.")

**Assignment:** Look at the output of the cell above. How do you think the friend groups are grouped? Do you understand why these heroes are in the same friend group?

# Partners
This material was developed by Dwengo vzw and was made possible with support from VLAIO. Find all the materials from our wAIsda project at [dwengo.org/waisda](dwengo.org/waisda)
!["VLAIO logo"](img/vlaio.png)
!["Dwengo logo"](img/dwengo-groen-zwart.png)